# Grounded VQG reproduction -- full training on Colab Pro

Reproduces *Automatic Generation of Grounded Visual Questions* (Zhang et al., IJCAI 2017), architecture-faithful, with library substitutions for deprecated/unreachable dependencies:
- Original Torch/Lua DenseCap -> [`soloist97/densecap-pytorch`](https://github.com/soloist97/densecap-pytorch) (only affects the diversity/quality of the *input* captions, not the VQG model architecture -- see this project's README for the full writeup).
- `densecap-pytorch`'s own pretrained checkpoint (OneDrive/BaiduYun) turned out to be unreachable -- broken share link on OneDrive, anonymous-download blocked on BaiduYun -- so this notebook **trains densecap-pytorch itself** on Visual Genome (cell 6), using a lightly patched `train.py` (native `torch.cuda.amp` instead of the unmaintained NVIDIA Apex dependency, plus checkpoint-resume support upstream doesn't have). No change to the model architecture or hyperparameters, just robustness for a multi-hour Colab job.
- Original NeuralTalk2 baseline -> not reproduced here (this notebook trains the paper's actual model only).

**Before running:** Runtime -> Change runtime type -> GPU (A100 recommended if your Colab Pro quota allows; T4 works, just slower).

**This will take hours, possibly across multiple sessions.** COCO images (~26GB) + Visual Genome (~15GB) + DenseCap training (10 epochs over ~108k images) + full VQG training over VQA v1 (764k questions) or Visual7W (327k QA pairs) is a genuinely long pipeline. Everything below writes intermediate artifacts to Google Drive so you can stop and resume -- the DenseCap training step in particular is designed to survive a disconnect and pick back up automatically.

## 0. Config -- fill these in

In [ ]:
GITHUB_REPO_URL = "https://github.com/malimustafaa/Automatic-Generation-of-Grounded-Visual-Questions.git"
DATASET = "vqa"  # "vqa" or "visual7w"
DRIVE_ROOT = "/content/drive/MyDrive/grounded-vqg-reproduction"  # all downloads/checkpoints persist here

# DenseCap-pytorch: we train this ourselves (see cell 6) rather than relying on the
# author's pretrained checkpoint, which turned out to be unreachable via both OneDrive
# (broken share link) and BaiduYun (blocks anonymous downloads over ~1GB). These paths
# are where OUR trained checkpoint/config end up -- nothing to fill in by hand here.
DENSECAP_MODEL_PARAMS_DIR = f"{DRIVE_ROOT}/densecap_pytorch/model_params"  # Drive-backed: survives disconnects
DENSECAP_MODEL_NAME = "train_all_val_all_bz_2_epoch_10_inject_init"
# Prefer the best-validation-mAP checkpoint (what the densecap-pytorch maintainer told
# users to actually use, per github.com/soloist97/densecap-pytorch/issues/2) over the
# final-epoch "_end" one; cell 6d falls back to "_end" only if "_best" was never written
# (possible if training finished before any 20k-iteration eval improved on it).
DENSECAP_CHECKPOINT_BEST = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}_best.pth.tar"
DENSECAP_CHECKPOINT_END = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}_end.pth.tar"
DENSECAP_CONFIG = f"{DENSECAP_MODEL_PARAMS_DIR}/{DENSECAP_MODEL_NAME}/config.json"

## 1. Mount Drive + clone this repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

# If the repo already exists (runtime didn't actually reset), `git pull` instead of
# `git clone` -- a bare clone silently fails with "destination path already exists"
# in that case, leaving whatever stale script versions were already on disk, which is
# exactly what caused a fix pushed to GitHub to not actually take effect after a
# "rerun all cells" that didn't go through a genuinely fresh runtime.
if os.path.exists("/content/grounded-vqg-reproduction"):
    print("Repo already present -- pulling latest instead of cloning.")
    !cd /content/grounded-vqg-reproduction && git pull
else:
    !git clone {GITHUB_REPO_URL} /content/grounded-vqg-reproduction
%cd /content/grounded-vqg-reproduction

## 2. Install dependencies

In [ ]:
!pip install -q pyyaml pycocoevalcap tqdm
# torch/torchvision/numpy/pillow are already present on Colab images.

# Run the smoke test first -- if this fails, nothing downstream will work either,
# and it takes seconds rather than hours to find out.
!python -m tests.smoke_test

## 3. Download GloVe + COCO images (persisted to Drive)

In [ ]:
!bash scripts/download_glove.sh {DRIVE_ROOT}/data

# COCO images extract to LOCAL disk (/content/coco), not the Drive mount -- reading/
# writing ~123k small files through Drive's FUSE layer is both slow and prone to
# sporadic I/O errors at that scale (confirmed in practice, including an attempted
# Drive->local rsync migration that turned out just as slow: rsync still has to open
# each file individually through the same FUSE mount, so bulk-copying doesn't dodge
# the bottleneck either -- a fresh network download + local unzip is dramatically
# faster than moving data off Drive by any method). If train2014/val2014 already exist
# on Drive from an earlier run, that's now dead weight you can delete whenever
# convenient -- it's not used anywhere in this pipeline.
import os
COCO_LOCAL = "/content/coco"
os.makedirs(COCO_LOCAL, exist_ok=True)
!bash scripts/download_coco_images.sh {COCO_LOCAL} {DRIVE_ROOT}/data/coco_zips

## 4. Download & flatten the question dataset (VQA v1 or Visual7W)

In [ ]:
if DATASET == "vqa":
    !python scripts/prepare_vqa.py --dest_dir {DRIVE_ROOT}/data/vqa_raw --out {DRIVE_ROOT}/data/questions.json
else:
    !python scripts/prepare_visual7w.py --dest_dir {DRIVE_ROOT}/data/visual7w_raw --coco_dir {DRIVE_ROOT}/data/coco --out {DRIVE_ROOT}/data/questions.json

## 5. Extract frozen VGG-16 image features (300-d, paper Sec 3.2/4.4)

In [ ]:
!python scripts/extract_image_features.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --image_root {COCO_LOCAL} \
  --out_dir {DRIVE_ROOT}/data/image_features

## 6. DenseCap: train it ourselves, then generate candidate captions

The author's pretrained checkpoint turned out to be unreachable (OneDrive: broken share link; BaiduYun: blocks anonymous downloads of this size). So instead: download Visual Genome, preprocess it with `densecap-pytorch`'s own `preprocess.py`, then train with our patched `train.py` (native AMP instead of the unmaintained NVIDIA Apex dependency, plus checkpoint-resume support the upstream script doesn't have -- no changes to model architecture or hyperparameters).

**This step alone can take many hours.** If your Colab session disconnects partway through training, just re-run the training cell (6c) -- it auto-detects the last saved epoch on Drive and continues from there instead of restarting. Everything else in this section (VG download, preprocessing) is safe to re-run too; each step skips work that's already done.

In [ ]:
# 6a. Clone densecap-pytorch and install its (non-Apex) dependencies
import os
if not os.path.exists("/content/densecap-pytorch"):
    !git clone https://github.com/soloist97/densecap-pytorch.git /content/densecap-pytorch
else:
    print("densecap-pytorch already cloned, skipping.")

!pip install -q h5py tensorboard prefetch_generator tqdm
# NOTE: upstream's README also lists NVIDIA Apex as a dependency -- we don't need it,
# our patched train.py (cell 6c) uses PyTorch's native torch.cuda.amp instead.
#
# prefetch_generator (used by their DataLoaderPFG) is missing from the README
# entirely -- there's no requirements.txt in the repo, so this list was built by
# grepping every import across all six files the training path touches, not from
# their docs.

# Patch 1: model/evaluator.py imports `from nlgeval.pycocoevalcap.meteor.meteor
# import Meteor` for METEOR-based validation during training. nlg-eval is NOT on
# PyPI (confirmed) and installing it from GitHub source pulls in Theano + a pinned
# old gensim for ONE class we need, for metrics we don't use. pycocoevalcap
# (already installed in cell 2) vendors the same underlying MS-COCO Meteor code
# (confirmed by diffing both files directly), so we patch the import instead of
# installing nlg-eval. The two versions clean up differently -- nlg-eval's Meteor
# has an explicit close() method, pycocoevalcap's relies on __del__ -- so that
# call needs removing too, or it just trades one crash for another.
evaluator_path = "/content/densecap-pytorch/model/evaluator.py"
with open(evaluator_path) as f:
    content = f.read()
content = content.replace(
    "from nlgeval.pycocoevalcap.meteor.meteor import Meteor",
    "from pycocoevalcap.meteor.meteor import Meteor",
)
content = content.replace(
    "        meteor_scorer.close()\n",
    "        # pycocoevalcap's Meteor cleans up via __del__, no explicit close() method\n",
)
with open(evaluator_path, "w") as f:
    f.write(content)
print("Patched model/evaluator.py to use pycocoevalcap instead of nlg-eval.")

# Patch 2: this codebase targets PyTorch ~1.4 (per its README), where
# pack_padded_sequence's `lengths` argument accepted a CUDA tensor. Modern PyTorch
# requires it on CPU specifically (even though the rest of the batch is legitimately
# on GPU) -- three call sites hit this, one in box_describer.py and two in
# roi_heads.py's caption_loss (which runs immediately after box_describer in the
# training forward pass, so fixing only one would just crash on the next line).
box_describer_path = "/content/densecap-pytorch/model/box_describer.py"
with open(box_describer_path) as f:
    content = f.read()
content = content.replace(
    "        rnn_input_pps = pack_padded_sequence(word_emb, lengths=cap_lens, batch_first=True, enforce_sorted=False)",
    "        rnn_input_pps = pack_padded_sequence(word_emb, lengths=cap_lens.cpu(), batch_first=True, enforce_sorted=False)",
)
with open(box_describer_path, "w") as f:
    f.write(content)

roi_heads_path = "/content/densecap-pytorch/model/roi_heads.py"
with open(roi_heads_path) as f:
    content = f.read()
content = content.replace(
    "    predict_pps = pack_padded_sequence(caption_predicts, caption_length, batch_first=True, enforce_sorted=False)",
    "    predict_pps = pack_padded_sequence(caption_predicts, caption_length.cpu(), batch_first=True, enforce_sorted=False)",
)
content = content.replace(
    "    target_pps = pack_padded_sequence(caption_gt[:, 1:], caption_length, batch_first=True, enforce_sorted=False)",
    "    target_pps = pack_padded_sequence(caption_gt[:, 1:], caption_length.cpu(), batch_first=True, enforce_sorted=False)",
)
with open(roi_heads_path, "w") as f:
    f.write(content)
print("Patched box_describer.py and roi_heads.py: pack_padded_sequence lengths -> CPU.")

In [ ]:
# 6b. Download Visual Genome + preprocess into densecap-pytorch's expected format.
# Extracts locally (not to Drive) for the same reason as the COCO download -- ~108k
# small image files unzipped onto Drive's FUSE mount is dramatically slower than local
# disk. Only the ~15GB zip files themselves are cached on Drive, so re-running this on
# a fresh session doesn't re-download them.
!bash scripts/download_visual_genome.sh /content/visual-genome {DRIVE_ROOT}/data/vg_zips

import os
os.makedirs("/content/densecap-pytorch/data", exist_ok=True)
!ln -sfn /content/visual-genome /content/densecap-pytorch/data/visual-genome

%cd /content/densecap-pytorch
!python preprocess.py \
  --region_data /content/visual-genome/region_descriptions.json \
  --image_data /content/visual-genome/image_data.json \
  --split_json info/densecap_splits.json \
  --pickle_output ./data/VG-regions-dicts-lite.pkl \
  --h5_output ./data/VG-regions-lite.h5
%cd /content/grounded-vqg-reproduction

In [ ]:
# 6c. Train (RESUME-SAFE: if the session disconnects, just re-run this cell --
# it auto-loads the last epoch checkpoint from Drive and continues).
import os
os.environ["DENSECAP_MODEL_PARAMS_DIR"] = DENSECAP_MODEL_PARAMS_DIR

%cd /content/densecap-pytorch
!cp /content/grounded-vqg-reproduction/scripts/densecap_train_patched.py .
!mkdir -p {DENSECAP_MODEL_PARAMS_DIR}
!python densecap_train_patched.py
%cd /content/grounded-vqg-reproduction

In [ ]:
# 6d. Generate candidate captions for every image using our trained checkpoint.
# Prefer the best-val-mAP checkpoint; fall back to the final-epoch one only if "_best"
# was never written (e.g. training completed before any 20k-iteration eval improved on it).
if os.path.exists(DENSECAP_CHECKPOINT_BEST):
    DENSECAP_CHECKPOINT = DENSECAP_CHECKPOINT_BEST
elif os.path.exists(DENSECAP_CHECKPOINT_END):
    print(f"No '_best' checkpoint found, falling back to '_end': {DENSECAP_CHECKPOINT_END}")
    DENSECAP_CHECKPOINT = DENSECAP_CHECKPOINT_END
else:
    raise AssertionError(
        f"Training hasn't finished yet -- neither {DENSECAP_CHECKPOINT_BEST} nor "
        f"{DENSECAP_CHECKPOINT_END} exists. Re-run cell 6c to continue training "
        "(it resumes automatically from the last epoch)."
    )

# Processes BOTH train2014 and val2014 -- VQA v1 references images from both splits,
# and build_manifest.py silently drops any question whose image has no candidates, so
# skipping a split doesn't error, it just quietly loses that data. --result_dir is on
# Drive (not /content) so each split's completed result.json survives a disconnect --
# re-running this cell after a crash on one split reuses the other split's finished
# result.json instead of redoing it (this is what cost ~3hrs once already).
!python scripts/run_densecap.py \
  --densecap_repo /content/densecap-pytorch \
  --config_json {DENSECAP_CONFIG} \
  --checkpoint {DENSECAP_CHECKPOINT} \
  --img_dirs {COCO_LOCAL}/train2014 {COCO_LOCAL}/val2014 \
  --result_dir {DRIVE_ROOT}/data/densecap_raw \
  --questions {DRIVE_ROOT}/data/questions.json \
  --out {DRIVE_ROOT}/data/densecap_candidates.json

## 7. Build the final training manifest

In [ ]:
!python scripts/build_manifest.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --features_dir {DRIVE_ROOT}/data/image_features \
  --candidates {DRIVE_ROOT}/data/densecap_candidates.json \
  --out {DRIVE_ROOT}/data/manifest.json

## 8. Train

Paper-given: batch size 64, 128 epochs (VQA) / 64 epochs (Visual7W) -- `configs/default.yaml` defaults to 128; pass `--epochs 64` for Visual7W. All hyperparameters the paper never specifies (learning rate, hidden sizes, etc.) live in that same config file with inline comments -- edit there, not here, if you want to sweep them.

In [ ]:
epochs_arg = "" if DATASET == "vqa" else "--epochs 64"
!python -m src.train \
  --config configs/default.yaml \
  --manifest {DRIVE_ROOT}/data/manifest.json \
  --glove {DRIVE_ROOT}/data/glove.840B.300d.txt \
  --out_dir {DRIVE_ROOT}/checkpoints \
  {epochs_arg}

## 9. Generate + evaluate

Reproduces the paper's Fig. 3 precision/recall sweep over N=1..6 generated questions per image (`eval/evaluate.py::sweep_num_questions`). Fill in a checkpoint path and a held-out slice of the manifest before running.

In [ ]:
import json, torch
from src.vocab import Vocab
from src.embeddings import build_embedding_matrix
from src.model import GroundedVQGModel
from src.bigram_lm import KneserNeyBigram
from src.dataset import tokenize
from src.generate import generate_questions
from eval.evaluate import sweep_num_questions, group_references_by_image

CKPT_PATH = f"{DRIVE_ROOT}/checkpoints/checkpoint_epoch128.pt"  # adjust to the epoch you want to evaluate
ckpt = torch.load(CKPT_PATH, map_location="cpu")
vocab = Vocab(); vocab.idx2word = ckpt["vocab"]; vocab.word2idx = {w: i for i, w in enumerate(vocab.idx2word)}

embedding = build_embedding_matrix(vocab, dim=300)  # shapes only; real weights load via state_dict below
model = GroundedVQGModel(embedding, vocab_size=len(vocab),
                          type_hidden=ckpt["cfg"]["type_selector_hidden"],
                          decoder_hidden=ckpt["cfg"]["decoder_hidden"])
model.load_state_dict(ckpt["model"])
model.eval()

with open(f"{DRIVE_ROOT}/data/manifest.json") as f:
    manifest = json.load(f)
eval_records = manifest[:500]  # small held-out slice for a first pass; widen once this runs cleanly

bigram_lm = KneserNeyBigram(discount=0.75).fit([tokenize(r["question"]) for r in manifest])

generated_pool, references = {}, {}
for r in eval_records:
    image_id = str(r["image_id"])
    import numpy as np
    feat = torch.from_numpy(np.load(r["image_feat_path"])).float()
    qs = generate_questions(model, vocab, bigram_lm, feat, r["candidates"],
                             num_questions=6, beta=ckpt["cfg"]["bigram_beta"])
    generated_pool.setdefault(image_id, []).extend(qs)
    references.setdefault(image_id, []).append(r["question"])

results = sweep_num_questions(references, generated_pool, max_n=6)
for n, scores in results.items():
    print(n, scores)